# R2 – Modelos de referencia (BGLR), los dos años (v6)
**Kernel: R.** Corre después de R1. Mismos 5 pliegues que los modelos de DL; en paralelo.

| Modelo | Kernel |
|---|---|
| BLUP_A | pedigrí |
| RKHS_SNP | K (gaussiano de SNP) |
| RKHS_DM, RKHS_DM6, RKHS_DM6AUC | gaussianos de cada versión de imágenes |
| **RKHS trimodal** | A + K + el conjunto de imágenes que elija la red híbrida (`IMG_TRIMODAL`, sección 3) |

Métricas: r, MSE, r dentro de ensayo y top 20 %, y tabla por pliegue.

In [ ]:
suppressPackageStartupMessages({ library(BGLR); library(parallel) })
Sys.setenv(OPENBLAS_NUM_THREADS = 1)
AÑOS <- c("Y21", "Y22"); NUCLEOS <- 12
NITER_UNI <- 6000; BURN_UNI <- 1500; NITER_MUL <- 30000; BURN_MUL <- 5000
SALIDA <- "Resultados_R_v6"; dir.create(SALIDA, showWarnings = FALSE); dir.create("bglr_tmp", showWarnings = FALSE)
DAT <- lapply(AÑOS, function(a) readRDS(paste0("Datos_entrenamiento/kernels_", a, ".rds"))); names(DAT) <- AÑOS

eta_de <- function(dat, ks) { ETA <- lapply(ks, function(k) { e <- dat$EVD[[k]]; keep <- e$values > 1e-7
  list(V = e$vectors[, keep], d = e$values[keep], model = "RKHS") }); names(ETA) <- ks; ETA }
r_dentro <- function(y, p, ens) { ok <- ens > 0
  mean(sapply(split(data.frame(r = y[ok], p = p[ok]), ens[ok]), function(g) cor(g$r, g$p))) }
top20 <- function(y, p, ens) { ok <- ens > 0
  mean(sapply(split(data.frame(r = y[ok], p = p[ok]), ens[ok]), function(g) { k <- ceiling(0.2 * nrow(g))
    length(intersect(order(-g$r)[1:k], order(-g$p)[1:k])) / k })) }

correr <- function(modelos) {          # modelos: lista nombre -> c(kernels); corre todos los años, modelos y pliegues en paralelo
  tareas <- expand.grid(fold = 1:5, modelo = names(modelos), año = AÑOS, stringsAsFactors = FALSE)
  costo <- sapply(tareas$modelo, function(m) length(modelos[[m]])); tareas <- tareas[order(-costo), ]
  sal <- mclapply(seq_len(nrow(tareas)), function(j) {
    a <- tareas$año[j]; m <- tareas$modelo[j]; i <- tareas$fold[j]; dat <- DAT[[a]]; ks <- modelos[[m]]
    te <- which(dat$fold == i); yNA <- dat$y; yNA[te] <- NA
    multi <- length(ks) > 1; set.seed(345322 + i)
    fm <- BGLR(y = yNA, ETA = eta_de(dat, ks), nIter = if (multi) NITER_MUL else NITER_UNI, burnIn = if (multi) BURN_MUL else BURN_UNI,
               verbose = FALSE, saveAt = paste0("bglr_tmp/", a, "_", m, "_f", i, "_"))
    v <- sapply(ks, function(k) fm$ETA[[k]]$varU * mean(diag(dat$KERNELS[[k]])))
    list(año = a, modelo = m, fold = i, te = te, pred = fm$yHat[te], pct = 100 * v / sum(v))
  }, mc.cores = NUCLEOS, mc.preschedule = FALSE)
  if (any(sapply(sal, inherits, "try-error"))) stop("Falló alguna tarea")
  filas <- list(); pliegue <- list()
  for (a in AÑOS) for (m in names(modelos)) {
    dat <- DAT[[a]]; p <- numeric(length(dat$y)); pct <- NULL
    for (s in Filter(function(o) o$año == a && o$modelo == m, sal)) { p[s$te] <- s$pred; pct <- rbind(pct, s$pct) }
    write.csv(data.frame(GID = dat$gids, Fold_Prueba = dat$fold, trial.no = dat$ensayo, y_Real = dat$y, y_Pred = p),
              file.path(SALIDA, paste0("pred_", a, "_", m, ".csv")), row.names = FALSE)
    rf <- sapply(1:5, function(i) cor(dat$y[dat$fold == i], p[dat$fold == i]))
    mf <- sapply(1:5, function(i) mean((dat$y[dat$fold == i] - p[dat$fold == i])^2))
    pliegue[[paste(a, m)]] <- data.frame(año = a, modelo = m, pliegue = 1:5, r = round(rf, 4), MSE = round(mf, 4))
    filas[[paste(a, m)]] <- data.frame(año = a, modelo = m, r = mean(rf), sd_r = sd(rf), MSE = mean(mf), sd_MSE = sd(mf),
      r_dentro = r_dentro(dat$y, p, dat$ensayo), top20 = top20(dat$y, p, dat$ensayo),
      var_pct = if (length(modelos[[m]]) > 1) paste(sprintf("%s %.0f", modelos[[m]], colMeans(pct)), collapse = " | ") else "")
  }
  list(tabla = do.call(rbind, filas), pliegue = do.call(rbind, pliegue))
}

## 1. Modelos unimodales

In [ ]:
UNI <- list(BLUP_A = "A", RKHS_SNP = "K", RKHS_DM = "DM", RKHS_DM6 = "DM6", RKHS_DM6AUC = "DM6AUC")
t0 <- Sys.time(); R_uni <- correr(UNI); cat(sprintf("%.1f min\n", as.numeric(difftime(Sys.time(), t0, units = "mins"))))
T <- R_uni$tabla; T[, 3:8] <- round(T[, 3:8], 4); print(T, row.names = FALSE)
write.csv(T, file.path(SALIDA, "tabla_referencia_unimodales.csv"), row.names = FALSE)
write.csv(R_uni$pliegue, file.path(SALIDA, "tabla_referencia_unimodales_por_pliegue.csv"), row.names = FALSE)

## 2. RKHS trimodal
A + SNP + DM6-AUC en los dos años (el mismo conjunto de imágenes que la rama del híbrido). No depende del DL.

In [ ]:
TRI <- list(`RKHS_tri_A+SNP+DM6AUC` = c("A", "K", "DM6AUC"))      # los dos años y los 5 pliegues a la vez (10 tareas)
t0 <- Sys.time(); R_tri <- correr(TRI); cat(sprintf("%.1f min\n", as.numeric(difftime(Sys.time(), t0, units = "mins"))))
T3 <- R_tri$tabla; T3[, 3:8] <- round(T3[, 3:8], 4); print(T3, row.names = FALSE)
write.csv(T3, file.path(SALIDA, "tabla_referencia_trimodal.csv"), row.names = FALSE)
write.csv(R_tri$pliegue, file.path(SALIDA, "tabla_referencia_trimodal_por_pliegue.csv"), row.names = FALSE)